# ProxyTrace 04. 주장 추출 (GPT)

03 노트북이 나눈 **주주의 말 / 회사의 말**에서 검증할 주장을 뽑아요.

| 필드 | 뜻 |
| --- | --- |
| `quote` | 위임장 원문 그대로 (LLM이 바꾸지 않게) |
| `claim_ko` | 한국어 한 문장 요약 |
| `kind` | 사실(fact) / 수치(number) / 의견·약속(opinion_commitment) / 요구(request) / 의혹 제기(allegation) |
| `checkable` | 외부 자료로 확인할 수 있는가 |
| `topic` | 주제 (ESG 태깅 때 쓸 분류) |
| `page_no` | 원문이 실제로 있는 쪽 (코드가 직접 찾아서 확인) |
| `in_board_summary` | 회사가 요약 bullet에 넣은 핵심 주장인가 |

결과: `data/claims/{TICKER}/{제출일}_claims.json`, `.csv`

In [ ]:
%pip install openai python-dotenv pandas -q

## 1. 설정
- `TEST_NUMBERS = None`: 전체 10개를 돌려요. 지시문을 고친 뒤 빠르게 시험하려면 `[4]`처럼 번호를 넣으세요.
- API 키는 입력창에 넣어요. 코드에 저장되지 않아요.

In [ ]:
import csv
import json
import os
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env")   # 노트북과 같은 폴더의 .env에서 OPENAI_API_KEY 읽기

# ▶ 01~03 노트북과 같은 티커
TICKER = "META"

# ▶ 모델: 주장 추출은 작은 모델로 충분해요. 쓸 수 있는 모델 이름으로 바꿔도 돼요.
MODEL = "gpt-4o-mini"

# ▶ None = 전체 10개. 지시문을 고친 뒤 시험할 때는 [4]처럼 번호를 넣으세요.
TEST_NUMBERS = [3]

# API 키: 환경변수 OPENAI_API_KEY가 없으면 입력창이 떠요 (코드에 저장 안 됨)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")
client = OpenAI()

T = TICKER.lower()
manifest = json.loads((Path("data/raw") / T / "manifest.json").read_text(encoding="utf-8"))
FILING = manifest["proxies"][0]["filingDate"]
parsed = json.loads((Path("data/parsed") / T / f"{FILING}_proposals.json").read_text(encoding="utf-8"))
pages = json.loads((Path("data/clean") / T / f"{FILING}_pages.json").read_text(encoding="utf-8"))["pages"]

CLAIM_DIR = Path("data/claims") / T
CLAIM_DIR.mkdir(parents=True, exist_ok=True)
print(f"{manifest['company']} {FILING}: 주주제안 {len(parsed['proposals'])}개")

## 2. 출력 형식 + 지시문
LLM이 항상 같은 형식(JSON)으로 답하게 강제해요. 지시문은 영어가 정확도가 더 높아서 영어로 썼어요.

In [ ]:
TOPICS = [
    "voting_governance",      # 투표·의결권·지배구조·이사회
    "compensation",           # 경영진 보수
    "privacy_ai_data",        # 개인정보·AI 데이터
    "human_rights_content",   # 인권·콘텐츠 정책·혐오
    "child_safety",           # 아동 안전
    "climate_energy",         # 기후·에너지·데이터센터
    "workforce_immigration",  # 인력·고용·이민
    "other",
]

CLAIM_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["claims"],
    "properties": {
        "claims": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["quote", "claim_ko", "kind", "checkable", "topic", "time_ref"],
                "properties": {
                    "quote":     {"type": "string", "description": "원문에서 그대로 복사한 문장 (수정 금지)"},
                    "claim_ko":  {"type": "string", "description": "주장을 한국어 한 문장으로"},
                    "kind":      {"type": "string", "enum": ["fact", "number", "opinion_commitment", "request", "allegation"]},
                    "checkable": {"type": "boolean", "description": "외부 자료로 사실 여부를 확인할 수 있는가"},
                    "topic":     {"type": "string", "enum": TOPICS},
                    "time_ref":  {"type": "string", "description": "주장이 가리키는 시점 (예: '2025 Annual Meeting'), 없으면 빈 문자열"},
                },
            },
        }
    },
}

SYSTEM = """You extract verifiable claims from a U.S. proxy statement for an investor fact-checking tool.
Goal: find the few statements an investor should check, not every sentence.

Rules:
- Split the text into atomic claims: one checkable statement per claim.
  If one sentence contains two different facts, make two claims that share the same quote.
  Example: "Mr. Zuckerberg has 60% voting power and votes all his shares to approve executive pay"
    → number: "저커버그는 의결권 60%를 가진다" / fact: "저커버그는 모든 주식을 보수 승인에 투표한다"
- "quote" MUST be copied exactly from the text (same words, same punctuation). Never paraphrase inside "quote".
- claim_ko must keep the SAME meaning as the quote. Do not add, drop or change facts.
- kind:
  - fact: something that exists or happened, with a specific, documentable subject (a named policy, committee, filing, event or date).
  - number: a statement whose core is a figure, percentage, count or amount.
  - opinion_commitment: beliefs, values, intentions, promises, judgments and interpretations.
  - request: what the proponent asks for, including the list of things a requested report should cover.
  - allegation: an accusation of wrongdoing, often hedged.
- Company self-description: when the company describes the quality of its own internal work
  ("we have robust practices", "we engage broadly", "we regularly meet", "we take steps to", "we are committed to"),
  use opinion_commitment with checkable false. Outsiders cannot see or measure this.
  Exception: if it names a specific, documentable item, it is a fact.
    "Our board oversees AI data practices primarily through the Audit & Privacy Committee" → fact, checkable true (committee charter)
    "We have robust risk review processes" → opinion_commitment, checkable false
- Interpretations are opinions: "Meta's strategy is to monetize...", "somewhat open source", "a major player" → opinion_commitment.
- Hidden facts: if a request, opinion or allegation contains a past fact or a figure, ALSO extract that fact as its own fact/number claim (same quote).
  The hidden-fact claim_ko states ONLY the fact.
  Example quote: "the same vote on executive pay that was on the 2025 Meta annual meeting ballot, to be on each annual meeting ballot"
    → request: "매년 주총 투표지에 경영진 보수 투표를 올리라고 요구한다", checkable false
    → fact: "2025년 Meta 주총 투표지에 경영진 보수 투표가 있었다", checkable true
  Example quote: "This is especially important at a company with a $1.8 trillion market capitalization"
    → number: "메타의 시가총액은 1.8조 달러이다", checkable true (the "important" part is not extracted)
- A statement about the future that follows from known facts ("the next vote will not be until 2028") is a fact claim (checkable true).
- No repeats: if the same fact appears more than once, extract it ONCE, using the quote from its FIRST appearance.
- checkable = true only if an outside document (SEC filing, company webpage, regulator or government data) could confirm or contradict it.
  opinion_commitment and request are always false. allegation is false unless it names a specific, documentable event.
  General statements that are not about this company (industry trends, how AI works in general) are false.
  A vague statement that only says evidence exists, without saying what it is, is false.
- Skip boilerplate (voting instructions, footnote URLs, "the board recommends a vote AGAINST").
- Write claim_ko in Korean.
"""


## 3. 함수 정의

In [ ]:
SPEAKERS = {"shareholder": ("proponent_text", "S"), "company": ("board_text", "C")}


def ws(s: str) -> str:
    """공백·따옴표 차이를 무시하고 비교하기 위한 정리"""
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", s).strip().lower()


PAGE_TEXT = [(p["page_no"], ws(p["text"])) for p in pages]


def find_page(quote: str, rng=None):
    """인용문이 실제로 위임장 몇 쪽에 있는지 찾기. 못 찾으면 None (= LLM이 문장을 바꿨을 가능성)
    rng(안건의 쪽 범위)가 있으면 그 안에서 먼저 찾아요 — 요약표에 같은 문장이 있어도 안건 쪽을 고르게."""
    q = ws(quote)
    in_rng = [(no, txt) for no, txt in PAGE_TEXT if rng and no and rng[0] <= no <= rng[1]]
    for pool in (in_rng, PAGE_TEXT):
        for key in (q, q[:80], q[:40]):
            if len(key) < 15:
                break
            for no, txt in pool:
                if key in txt:
                    return no
    return None


def lead_proponent(s: str) -> str:
    """'lead filer A and co-filers B, C' → A (대표 제안자만)"""
    if not s:
        return s
    m = re.search(r"(?:co-)?lead filer\s+(.+?)(?:,?\s+and\s+(?:co-lead filer|co-filers?)|,\s*co-lead filer|$)", s, re.I)
    name = m.group(1) if m else s
    return re.sub(r"^the\s+", "", name.strip(" ,."), flags=re.I)


def extract(text: str, title: str, speaker: str) -> tuple[list[dict], dict]:
    who = "the shareholder proponent" if speaker == "shareholder" else "the company's board of directors"
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Proposal: {title}\\nSpeaker: {who}\\n\\nText:\\n{text}"},
        ],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "claims", "schema": CLAIM_SCHEMA, "strict": True}},
    )
    data = json.loads(resp.choices[0].message.content)
    u = resp.usage
    return data["claims"], {"input": u.prompt_tokens, "output": u.completion_tokens}


def run(proposal: dict) -> tuple[list[dict], dict]:
    out, usage = [], {"input": 0, "output": 0}
    summary = [ws(b) for b in proposal["board_summary"]]
    for speaker, (field, code) in SPEAKERS.items():
        claims, u = extract(proposal[field], proposal["title"], speaker)
        usage = {k: usage[k] + u[k] for k in usage}
        for i, c in enumerate(claims, 1):
            page = find_page(c["quote"], proposal["pages"])
            out.append({
                "claim_id": f"P{proposal['number']:02d}-{code}{i:02d}",
                "proposal": proposal["number"],
                "proposal_topic": proposal["topic"],
                "speaker": speaker,
                "proponent": lead_proponent(proposal["proponent"]),
                **c,
                "page_no": page,
                "quote_found": page is not None,
                # 회사가 요약 bullet에 직접 넣은 주장인가 (= 회사가 강조한 핵심 주장)
                "in_board_summary": speaker == "company" and any(ws(c["quote"])[:60] in s or s[:60] in ws(c["quote"]) for s in summary),
            })
    return out, usage

## 4. 실행

In [ ]:
targets = [p for p in parsed["proposals"] if TEST_NUMBERS is None or p["number"] in TEST_NUMBERS]
all_claims, total = [], {"input": 0, "output": 0}
for p in targets:
    claims, u = run(p)
    all_claims += claims
    total = {k: total[k] + u[k] for k in total}
    print(f"[{p['number']:>2}] {p['topic'][:40]:40} 주장 {len(claims):>3}개  (토큰 in {u['input']:,} / out {u['output']:,})")
print(f"\n합계: 주장 {len(all_claims)}개, 토큰 in {total['input']:,} / out {total['output']:,}")

## 5. 결과 확인 ⚠ 중요
- **`quote_found`가 False인 주장**: LLM이 원문을 바꿔 쓴 것 → 프롬프트를 고치거나 버려야 해요
- `kind`, `checkable`이 상식적인지
- 같은 내용이 너무 잘게 쪼개지거나, 반대로 여러 개가 한 주장에 뭉쳐 있지 않은지

In [ ]:
for c in all_claims:
    flag = "" if c["quote_found"] else "  ⚠ 원문에서 못 찾음"
    star = "★" if c["in_board_summary"] else " "
    chk = "검증가능" if c["checkable"] else "검증불가"
    print(f"{c['claim_id']} {star} p.{c['page_no']} | {c['speaker']:11} | {c['kind']:18} | {chk} | {c['topic']}{flag}")
    print(f"    {c['claim_ko']}")
    print(f"    \"{c['quote'][:150]}\"")

In [ ]:
n = len(all_claims) or 1
print(f"원문 확인: {sum(c['quote_found'] for c in all_claims)}/{n}")
print(f"검증 가능: {sum(c['checkable'] for c in all_claims)}/{n}")
print(f"회사 요약 bullet 주장(★): {sum(c['in_board_summary'] for c in all_claims)}")

## 6. 저장
JSON은 다음 노트북용, CSV는 엑셀로 열어서 직접 확인·정답 라벨 붙이기용이에요.

In [ ]:
suffix = "" if TEST_NUMBERS is None else "_test"
jpath = CLAIM_DIR / f"{FILING}_claims{suffix}.json"
jpath.write_text(json.dumps({"ticker": TICKER, "filingDate": FILING, "model": MODEL,
                             "usage": total, "claims": all_claims}, ensure_ascii=False, indent=2), encoding="utf-8")

cpath = CLAIM_DIR / f"{FILING}_claims{suffix}.csv"
if all_claims:
    with cpath.open("w", newline="", encoding="utf-8-sig") as f:   # utf-8-sig: 엑셀에서 한글 안 깨지게
        w = csv.DictWriter(f, fieldnames=list(all_claims[0].keys()))
        w.writeheader()
        w.writerows(all_claims)
print(f"저장: {jpath}\n저장: {cpath}")

## 다음 단계
4번 안건 결과(5번 셀)를 보내 주세요. 특히 아래가 제대로 나왔는지 봐요.
- 회사 주장 **"2025년 주총에서 3년 주기가 승인됐다"** → `fact`, 검증 가능, ★
- "우리는 3년 주기가 적절하다고 믿는다" → `opinion_commitment`, 검증 불가

괜찮으면 `TEST_NUMBERS = None`으로 전체를 돌리고, `05_esg_priority.ipynb`(ESG 태깅 + 우선순위)로 넘어가요.